# 83. Olist电商物流履约分析

<!-- module-learning-arc:start -->
> **综合项目 模块主线｜第 2 / 4 步：从多表数据诊断履约问题**
>
> **持续应用背景：** 进入数据分析决策实验室：连续处理客户价值、物流履约、供需调度和营销资源四类问题，训练从业务问题到行动建议的迁移能力。
>
> **承接上一阶段：** 在线零售用户消费与RFM  →  **本章任务：** Olist电商物流履约分析  →  **下一步：** 共享单车需求与运力调度
>
> **大作业连接：** 本章练习将成为《跨模块业务决策项目》的一部分，最终需要把前四个项目形成的方法迁移为项目提案、最短充分证据链和决策备忘录。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：Olist 是巴西的一家电商平台，把数以万计的中小卖家接进同一个商城。发货准不准、客户多久能收到货，直接决定平台口碑和退款率。这一章我们从读取订单与商品明细入手，连表、算时效、定位延期卖家，完整走一遍电商物流履约的诊断流程。



## 本章目标

学完本章，你将能够：

- **理解**：理解「Olist电商物流履约分析」的核心概念、适用场景与关键口径。
- **操作**：能按本章步骤写出可复现的实现，并读懂输出/结果。
- **迁移**：能用本章方法处理一份新数据，独立完成同类任务并给出结论。


## 83.1 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| order_id | 订单编号 | 订单表与明细表连接键 |
| order_status | 订单状态 | delivered/canceled 等 |
| order_purchase_timestamp | 下单时间 | 流程起点 |
| order_delivered_carrier_date | 交承运商时间 | 出库完成 |
| order_delivered_customer_date | 客户签收时间 | 实际完成 |
| order_estimated_delivery_date | 预计送达时间 | 承诺基线 |
| price/freight_value | 商品价/运费 | 订单明细金额 |
| seller_id | 卖家编号 | 履约责任维度 |

## 83.2 数据质量检查清单

- 订单主键是否唯一
- 订单明细一对多连接后是否重复计算订单
- 各里程碑时间缺失与逆序
- 未签收订单不能计算实际履约时长
- 异常负时长和极端长尾
- 金额字段是否非负


## 83.3 项目任务

1. 读取并连接订单与明细
2. 聚合到订单粒度避免重复
3. 构造出库、运输、总履约与延期指标
4. 分析月度 SLA
5. 定位卖家延期集中度
6. 输出物流运营行动清单


## 83.4 项目阶段速查

先看每个阶段要做什么、留下什么证据，再按任务顺序运行项目代码。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 读取订单与商品明细 | `pd.read_csv()`、`orders.order_id.duplicated()`、`orders.order_id.isin()`、`orders.order_status.value_counts()` | 订单表是一单一行，商品表是一单多行；先分别审计，再聚合连接。 | 订单主键是否唯一 |
| 2. 构造订单级物流宽表 | `items.groupby()`、`seller_id.first()`、`orders.merge()`、`dt.total_seconds()` | 先把商品价、运费和卖家数聚合到订单级，避免多商品订单把时效重复计权。 | 订单明细一对多连接后是否重复计算订单 |
| 3. SLA与流程瓶颈 | `logistics.order_delivered_customer_date.notna()`、`delivered.order_purchase_timestamp.dt.to_period()`、`delivered.groupby()`、`x.quantile()` | 只在已签收订单上衡量实际时效；分别看出库和运输才能定位责任环节。 | 各里程碑时间缺失与逆序 |
| 4. 卖家与运费风险 | `delivered.groupby()`、`seller.query()`、`eligible.head()`、`delivered.freight_ratio.median()` | 设置最小订单量，避免用少量订单给卖家贴标签。高延期率只是筛查信号，还需拆分承运商和地区。 | 未签收订单不能计算实际履约时长 |
| 5. 物流运营结论 | `monthly.late_rate.idxmax()`、`delivered.lead_days.quantile()` | 形成可执行的 SLA 分层、监控和后续数据需求。 | 异常负时长和极端长尾 |


## 83.5 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 83.6 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 83.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 83.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 83.9 读取订单与商品明细

订单表是一单一行，商品表是一单多行；先分别审计，再聚合连接。


<!-- math-foundation:chapter-83 -->
### 数学推导｜履约异常率与平均延迟

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把日期差转成同一单位。** 每单延迟天数为 $d_i=t_i^{actual}-t_i^{promised}$。

**第 2 步｜把“是否延期”变成指标。** $l_i=\mathbf{1}(d_i>0)$，因此延期订单数是 $\sum_i l_i$。

**第 3 步｜区分两个不同问题。** $r_{late}=\sum_i l_i/n$ 回答“多少订单延期”，而

$$
\bar d_{late}=\frac{\sum_i d_i l_i}{\sum_i l_i}
$$

回答“已延期订单平均晚几天”；它与全体订单的平均日期差不是同一口径。

**把上面的关系收束为本章计算式：**

$$
r_{late}=\frac{\sum_i\mathbf{1}(d_i>0)}{n},\qquad \bar{d}=\frac{1}{n}\sum_{i=1}^{n}d_i
$$

**符号解释：** $d_i$ 是实际送达日减预计送达日的天数，$r_{late}$ 是延期订单占比。

**代码对应：** 构造 `delay_days` 和 `is_late` 后按月份、地区或品类分组聚合。

**使用边界：** 未送达订单与缺失预计日期需要单独定义，不能静默排除后仍称为总体延期率。


In [ ]:
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import pandas as pd
import numpy as np

# 中文字体支持：自动选用可用的中文字体，避免图表中文显示为方框
if os.path.exists("/tmp/NotoSansSC-Regular.otf"):
    fm.fontManager.addfont("/tmp/NotoSansSC-Regular.otf")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(
    "/datasets/olist_orders_dataset.csv", parse_dates=date_cols
)
items = pd.read_csv(
    "/datasets/olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"],
)
print("订单/明细:", orders.shape, items.shape)
print(
    "订单主键重复:",
    orders.order_id.duplicated().sum(),
    " 无明细订单:",
    (~orders.order_id.isin(items.order_id)).sum(),
)
print("状态分布:\n", orders.order_status.value_counts())


**练一练**：上一步已经读入订单表 `orders` 和明细表 `items`。质量审计的第一步是量化问题规模——请在 `orders` 上统计「客户签收时间」`order_delivered_customer_date` 缺失的订单数量。这些订单尚未送达或中途被取消，不能参与后续「实际履约时长」的计算，务必在动手算时效前把它们数清楚。用 `isna()` 配合 `sum()` 即可完成。


In [ ]:
# 请在下方填写代码
# 目标：统计 orders 中 order_delivered_customer_date 缺失的订单数量
# 提示：orders['order_delivered_customer_date'].isna().sum()
missing_sign = ____
print("未签收(签收时间缺失)订单数:", missing_sign)


In [ ]:
# 完整答案
missing_sign = orders["order_delivered_customer_date"].isna().sum()
print("未签收(签收时间缺失)订单数:", missing_sign)


## 83.10 构造订单级物流宽表

先把商品价、运费和卖家数聚合到订单级，避免多商品订单把时效重复计权。


In [ ]:
item_order = items.groupby("order_id").agg(
    item_count=("order_item_id", "size"),
    goods_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
    seller_count=("seller_id", "nunique"),
)
seller_order = (
    items.groupby("order_id").seller_id.first().rename("primary_seller")
)
logistics = orders.merge(item_order, on="order_id", how="left").merge(
    seller_order, on="order_id", how="left", validate="one_to_one"
)
logistics["dispatch_days"] = (
    logistics.order_delivered_carrier_date - logistics.order_purchase_timestamp
).dt.total_seconds() / 86400
logistics["transit_days"] = (
    logistics.order_delivered_customer_date
    - logistics.order_delivered_carrier_date
).dt.total_seconds() / 86400
logistics["lead_days"] = (
    logistics.order_delivered_customer_date
    - logistics.order_purchase_timestamp
).dt.total_seconds() / 86400
logistics["delay_days"] = (
    logistics.order_delivered_customer_date
    - logistics.order_estimated_delivery_date
).dt.total_seconds() / 86400
logistics["late"] = logistics.delay_days.gt(0)
logistics["freight_ratio"] = (
    logistics.freight_value / logistics.goods_value.replace(0, np.nan)
)
print(
    logistics[
        [
            "dispatch_days",
            "transit_days",
            "lead_days",
            "delay_days",
            "freight_ratio",
        ]
    ]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)


## 83.11 SLA与流程瓶颈

只在已签收订单上衡量实际时效；分别看出库和运输才能定位责任环节。


In [ ]:
# 中文字体支持：避免图表中文显示为方框
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

delivered = logistics[
    (logistics.order_status == "delivered")
    & logistics.order_delivered_customer_date.notna()
].copy()
valid = (
    (delivered.dispatch_days >= 0)
    & (delivered.transit_days >= 0)
    & (delivered.lead_days >= 0)
)
delivered = delivered.loc[valid]
delivered["purchase_month"] = delivered.order_purchase_timestamp.dt.to_period(
    "M"
).astype(str)
monthly = delivered.groupby("purchase_month").agg(
    orders=("order_id", "size"),
    late_rate=("late", "mean"),
    median_dispatch=("dispatch_days", "median"),
    median_transit=("transit_days", "median"),
    p90_lead=("lead_days", lambda x: x.quantile(0.9)),
)
print(
    "已签收有效订单:",
    len(delivered),
    " 延期率:",
    f"{delivered.late.mean():.1%}",
)
print(monthly.tail(12).round(2))
monthly.late_rate.plot(figsize=(10, 4), marker="o", title="按下单月的延期率")
plt.ylabel("延期率")
plt.tight_layout()
plt.show()


## 83.12 卖家与运费风险

设置最小订单量，避免用少量订单给卖家贴标签。高延期率只是筛查信号，还需拆分承运商和地区。


In [ ]:
seller = delivered.groupby("primary_seller").agg(
    orders=("order_id", "size"),
    late_rate=("late", "mean"),
    median_dispatch=("dispatch_days", "median"),
    median_freight_ratio=("freight_ratio", "median"),
)
eligible = seller.query("orders>=30").sort_values(
    ["late_rate", "orders"], ascending=False
)
print("订单>=30的高延期卖家:\n", eligible.head(12).round(3))
print(
    "运费占商品价值中位数:",
    f"{delivered.freight_ratio.median():.1%}",
    " P90:",
    f"{delivered.freight_ratio.quantile(.9):.1%}",
)


## 83.13 物流运营结论

形成可执行的 SLA 分层、监控和后续数据需求。


In [ ]:
worst = monthly.late_rate.idxmax()
p90 = delivered.lead_days.quantile(0.9)
print(f"1. 月度延期峰值出现在 {worst}，回查当月卖家出库与承运时长。")
print(f"2. 总履约 P90 为 {p90:.1f} 天，建议按地区/品类进一步建立差异化承诺。")
print(
    f"3. 将 {len(eligible)} 个订单量>=30的卖家纳入分层SLA看板，重点看延期率与出库中位数。"
)
print(
    "限制：公开数据不含承运商、仓库节点和实时轨迹；卖家差异可能受地区与品类结构影响，不能直接归因。"
)


## 83.14 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 83.14.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 83.14.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 83.15 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 83.15.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 83.16 易错点提醒

**易错点 1**：orders 与 order_items 是 1 对多关系，直接合并会把订单重复 N 行；先确认聚合粒度（订单级先 groupby，商品级再合并）。

**易错点 2**：时间列带时区（UTC），本地比较前先统一，否则"履约时长"会整体偏移几小时。

**易错点 3**：取消/不可用订单（order_status）不算正常履约，算延期或时长前先过滤或单独标记。

**易错点 4**：四张表的关联键（order_id / customer_id / seller_id）可能缺值或重复；连接前先检查唯一性。

**易错点 5**：承诺时间（order_estimated_delivery_date）与签收时间单位不一致（日期 vs 带时间戳）时先统一格式再相减。


## 83.17 结论与表达

- 物流时效必须在订单粒度计算。
- 延期要拆成出库和在途环节。
- 卖家排名必须设置样本量门槛并做结构校正。
- 公开订单数据适合诊断，不足以做承运商因果评估。


## 83.18 项目验收清单

- 连接后保持一单一行
- 能解释未签收订单的处理
- 能计算月度延期率和 P90
- 建议包含 SLA、责任环节和数据限制

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 83.19 小结

使用 Olist 巴西电商近 10 万笔真实公开订单，连接订单与商品明细，诊断承运时效、延期风险、运费负担和卖家履约表现。


### 83.19.1 你已经完成

- 建立订单级物流宽表
- 正确处理未签收与时间缺失
- 衡量采购到发货、在途和总履约时长
- 识别延期订单的时间与卖家集中度
- 把诊断转化为物流 SLA 与监控建议


### 83.19.2 质量与结论提醒

- 订单主键是否唯一
- 订单明细一对多连接后是否重复计算订单
- 各里程碑时间缺失与逆序
- 物流时效必须在订单粒度计算。
- 延期要拆成出库和在途环节。
- 卖家排名必须设置样本量门槛并做结构校正。
- 公开订单数据适合诊断，不足以做承运商因果评估。


### 83.19.3 项目交付检查

- [ ] 连接后保持一单一行
- [ ] 能解释未签收订单的处理
- [ ] 能计算月度延期率和 P90
- [ ] 建议包含 SLA、责任环节和数据限制


### 83.19.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
